# Understanding retrieval augmented generation (RAG) 🧠

## What you will learn in this course 🧐🧐

LLMs know a lot, but they do not know *your* data. They cannot answer questions about your company's help center, your product documentation, or your internal policies. **Retrieval Augmented Generation** (RAG) is the technique that solves this. Before the LLM answers a question, RAG finds the relevant documents and passes them to the model. The model reads them and answers based on what it found.

Because of this, you need to understand the **five building blocks of RAG**. This lecture explains each one with small Python demos so you can see exactly what happens at every step. You will not need to write this code yourself in production. n8n handles it for you. But understanding these concepts will help you configure n8n's RAG nodes correctly and debug problems when they appear.

By the end of this lecture, you will:

- Build a complete RAG pipeline from ingestion to generation using Python
- Apply three chunking strategies to real documents and evaluate their trade-offs
- Use an embedding model to convert text into numerical vectors and measure similarity between them
- Store and search document embeddings in a vector database with metadata filters
- Construct a RAG prompt that combines retrieved context with a user question for accurate LLM answers

## Why RAG exists

<img src="https://ai-essentials-assets.s3.eu-west-3.amazonaws.com/M03-AI_Applications_%26_Automation/AIE-M03-D02-Logo_CloudSync.png" width="300" />

Let's start with a concrete example. Imagine you manage customer support at a SaaS company called **CloudSync**. Your team answers 200 tickets per day. Most answers already exist somewhere: in your help center, API docs, or runbooks. But agents spend 40% of their time *searching* for the right answer.

You want an AI assistant that reads a customer question, finds the right documentation, and drafts an answer. You could try to fine-tune an LLM on your documents, but that is expensive, slow, and hard to update. Every time a help article changes, you would need to retrain the model.

RAG takes a different approach. It works in two phases:

<img src="https://ai-essentials-assets.s3.eu-west-3.amazonaws.com/M03-AI_Applications_%26_Automation/AIE-M03-D02-RAG_pipeline.png" />

**Phase 1, ingestion (done once).** You take your documents, split them into small pieces, convert each piece into numbers (an **embedding**), and store everything in a special database. This phase is slow, but you only do it once. When your documents change, you update the database with the new pieces. In this phase, you prepare the information for the LLM, but the LLM is not involved yet.

**Phase 2, query time (done for every question).** When a user asks something, you convert the question into numbers too, find the most similar pieces in the database, and pass them to the LLM along with the question. The LLM reads the retrieved pieces and answers based on that information. This phase is fast, and it always uses the most up-to-date documents. In this phase, the LLM is fully involved. It reads the retrieved context and generates the answer.

> The core idea: do not ask the LLM to *remember*. Ask the LLM to *read and answer*.

## Setup

This lecture uses two libraries. **sentence-transformers** is a Python library that converts text into embeddings (lists of numbers that capture meaning). **chromadb** is a lightweight vector database that stores embeddings and lets you search them by similarity. Both run locally on your machine.

In [29]:
# !pip install chromadb sentence-transformers
import warnings
warnings.filterwarnings('ignore')

In [30]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

## Step 1: ingestion (loading your documents)

Ingestion means collecting all the documents your system will use to answer questions. In a real company, these come from many sources: help center articles, PDFs, Notion pages, Confluence wikis, Google Docs.

For this demo, we use five short CloudSync support documents stored as Python strings. In n8n, you would connect a Google Drive node, a Notion node, or an HTTP node to fetch documents automatically.

In [31]:
# CloudSync knowledge base: 5 support documents
# Each document has a title, a source label, and the text content

documents = [
    {
        "title": "CloudSync API basics",
        "source": "api_docs",
        "content": (
            "All API requests need an API key from Settings > API Keys. "
            "The base URL is https://api.cloudsync.io/v2/. "
            "Rate limits are 1000 requests per minute for Business plans "
            "and 100 per minute for Starter plans. "
            "If you exceed the limit, you get HTTP 429."
        )
    },
    {
        "title": "Sync conflicts",
        "source": "help_center",
        "content": (
            "A sync conflict happens when the same file is edited on two devices "
            "before the changes are synchronized. "
            "The default policy is last-write-wins. "
            "You can change it to keep-both or manual review. "
            "Go to Dashboard > Sync Status > Conflicts to see unresolved conflicts. "
            "Enable real-time collaboration mode to prevent conflicts on shared folders."
        )
    },
    {
        "title": "Pricing and plans",
        "source": "help_center",
        "content": (
            "Starter plan: $9/user/month with 50GB storage. "
            "Business plan: $29/user/month with 500GB storage, API access, and SSO. "
            "Enterprise plan: custom pricing with unlimited storage. "
            "Annual billing is default. Monthly billing costs 20% more. "
            "Educational institutions get 40% off the Business plan."
        )
    },
    {
        "title": "Escalation procedures",
        "source": "internal_runbook",
        "content": (
            "Tier 1 handles password resets and basic sync issues. "
            "Escalate to Tier 2 for data loss or API 500 errors. "
            "Escalate to Tier 3 (engineering) if more than 10 users are affected "
            "or if an Enterprise customer reports SLA downtime. "
            "Never promise a specific resolution time unless it is in the SLA."
        )
    },
    {
        "title": "SSO setup with SAML",
        "source": "api_docs",
        "content": (
            "SSO is available on Business and Enterprise plans. "
            "Create a SAML app in your identity provider (Okta, Azure AD, Google Workspace). "
            "Use https://auth.cloudsync.io/saml/callback as the ACS URL. "
            "Upload the IdP metadata XML in Settings > Security > SSO. "
            "If users see SAML response invalid, check the clock skew between servers."
        )
    }
]

print(f"Loaded {len(documents)} documents:")
for doc in documents:
    char_count = len(doc["content"])
    print(f"  - {doc['title']} ({char_count} chars, source: {doc['source']})")

Loaded 5 documents:
  - CloudSync API basics (245 chars, source: api_docs)
  - Sync conflicts (339 chars, source: help_center)
  - Pricing and plans (288 chars, source: help_center)
  - Escalation procedures (290 chars, source: internal_runbook)
  - SSO setup with SAML (322 chars, source: api_docs)


Each document has a `source` label: `api_docs`, `help_center`, or `internal_runbook`. This is **metadata**. It becomes very important later when we want to control which documents the AI can access. A public chatbot should never show internal runbooks to customers.

<Note type="important">

In production, ingestion is not a one-time event. When a help article changes, its data in the **vector database** must be updated too. n8n can schedule this with a Cron trigger that re-ingests documents every night.

</Note>

## Step 2: chunking (splitting documents into pieces)

Once you have your documents, you need to split them into smaller pieces called **chunks**. Why? Because LLMs have a **context window**, the maximum amount of text they can read at once. Even if the window is large, stuffing too much text into the prompt makes the model lose focus. Thanks to chunking, you can retrieve only the most relevant pieces of information instead of entire documents.

### Three common strategies

<img src="https://ai-essentials-assets.s3.eu-west-3.amazonaws.com/M03-AI_Applications_%26_Automation/AIE-M03-D02-Chunking_Strategies.png" />

1. **Fixed-size chunking** splits text every N characters. Simple but it cuts sentences in half.

2. **Overlapping chunking** is the same thing but chunks overlap by a few characters. Information at the boundary appears in both chunks, so nothing gets lost.

3. **Sentence-aware chunking** groups complete sentences together until a size limit is reached. No sentence is ever cut in the middle. This is the best default choice.

There are also **document-type heuristics** (rules for specific formats). For example, split Markdown on headings. Split HTML on `<h2>` tags. Split code on functions. These work well for structured content.

<Note type="tip">

There is no perfect chunking strategy. Start with sentence-aware chunks of 300 to 500 characters. Then test on real questions and adjust. n8n's built-in text splitter uses a recursive strategy that works well for most cases.

</Note>

Let us see the difference between **fixed-size and sentence-aware chunking** on the same text.

In [32]:
# The text we will chunk
sample_text = documents[0]["content"]
print(f"Original text ({len(sample_text)} chars):")
print(sample_text)
print()

Original text (245 chars):
All API requests need an API key from Settings > API Keys. The base URL is https://api.cloudsync.io/v2/. Rate limits are 1000 requests per minute for Business plans and 100 per minute for Starter plans. If you exceed the limit, you get HTTP 429.



In [33]:
# Fixed-size chunking: splits every 100 characters, ignores sentence boundaries
chunk_size = 100
fixed_chunks = []
start = 0

while start < len(sample_text):
    end = start + chunk_size
    chunk = sample_text[start:end]
    fixed_chunks.append(chunk)
    start = end

print(f"Fixed-size chunks ({len(fixed_chunks)} chunks):")
for i, chunk in enumerate(fixed_chunks):
    print(f"  Chunk {i + 1}: '{chunk}'")

Fixed-size chunks (3 chunks):
  Chunk 1: 'All API requests need an API key from Settings > API Keys. The base URL is https://api.cloudsync.io/'
  Chunk 2: 'v2/. Rate limits are 1000 requests per minute for Business plans and 100 per minute for Starter plan'
  Chunk 3: 's. If you exceed the limit, you get HTTP 429.'


Look at the output. Some chunks end in the middle of a sentence. "The base URL is https://api.cloudsync.io/'. is not a useful chunk. The URL is cut. The information is split between two chunks.

In [34]:
# Sentence-aware chunking: keeps complete sentences together

raw_sentences = sample_text.split(". ")

# Add the period back to each sentence except the last one
sentences = []
for i, s in enumerate(raw_sentences):
    is_last = (i == len(raw_sentences) - 1)
    if is_last:
        sentences.append(s)
    else:
        sentences.append(s + ".")

# Group sentences into chunks without exceeding the limit
max_size = 150
sentence_chunks = []
current_chunk = ""

for sentence in sentences:
    combined = current_chunk + " " + sentence
    combined = combined.strip()
    
    if len(combined) <= max_size:
        current_chunk = combined
    else:
        if current_chunk:
            sentence_chunks.append(current_chunk)
        current_chunk = sentence

# Add the last chunk
if current_chunk:
    sentence_chunks.append(current_chunk)

print(f"Sentence-aware chunks ({len(sentence_chunks)} chunks):")
for i, chunk in enumerate(sentence_chunks):
    print(f"  Chunk {i + 1}: '{chunk}'")

Sentence-aware chunks (2 chunks):
  Chunk 1: 'All API requests need an API key from Settings > API Keys. The base URL is https://api.cloudsync.io/v2/.'
  Chunk 2: 'Rate limits are 1000 requests per minute for Business plans and 100 per minute for Starter plans. If you exceed the limit, you get HTTP 429.'


Every chunk contains complete sentences. "Rate limits are 1000 requests per minute for Business plans and 100 per minute for Starter plans." stays in one piece. This chunk is a perfect answer to a rate limit question.

<Note type="tip">

In n8n, the **Recursive Character Text Splitter** sub-node does this automatically. You set the chunk size and overlap, and it handles sentence boundaries for you.

</Note>

## Step 3: embeddings (turning text into numbers)

An **embedding** is a list of numbers that represents the meaning of a text. It is a **vector**, a point in a high-dimensional space. Texts with similar meanings produce vectors that are close together.

For example, the sentence "How do I set up SSO (single sign-on)?" and "I need help configuring single sign-on" use different words but mean the same thing. Their embeddings will be very similar. This is why embedding-based search is better than keyword search.

<img src="https://ai-essentials-assets.s3.eu-west-3.amazonaws.com/M03-AI_Applications_%26_Automation/AIE-M03-D02-Embedding_space.png" />

An **embedding model** is a small AI model trained specifically for this job. It reads text and outputs a fixed-length list of numbers (for example, 384 numbers). We use `all-MiniLM-L6-v2`, a popular open-source model. In n8n, you can also use OpenAI's `text-embedding-3-small` or other providers.

Let us see embeddings in action.

In [35]:
# Load the embedding model (downloads ~80MB on first run)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed a single sentence
test_sentence = "How do I configure SSO?"
embedding = model.encode(test_sentence)

print(f"Input:  '{test_sentence}'")
print(f"Output: a list of {len(embedding)} numbers")
print(f"First 5 numbers: {embedding[:5]}")

Input:  'How do I configure SSO?'
Output: a list of 384 numbers
First 5 numbers: [ 0.03002776 -0.05543963 -0.15313922  0.00574375 -0.0375725 ]


The model turned our sentence into 384 numbers. You cannot interpret individual numbers.
What matters is the **distance** between different embeddings. Let us compare three sentences.
Let's use the **cosine similarity** formula to measure how close the embeddings are. The closer to 1, the more similar the meanings.

<Note type="important">

**[Cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity)** measures how similar two vectors are in direction. A score of 1.0 means identical meaning. A score of 0.0 means completely unrelated. In RAG, higher similarity means the document is more relevant to the question.

</Note>


In [36]:
sentence_a = "How do I set up single sign-on?"
sentence_b = "I need help configuring SSO with Okta"
sentence_c = "What are your pricing plans?"

# Convert each sentence into a vector of numbers
emb_a = model.encode(sentence_a)
emb_b = model.encode(sentence_b)
emb_c = model.encode(sentence_c)

def cosine_sim(v1, v2):
    # Multiply corresponding entries and sum them
    dot = np.dot(v1, v2)

    # Compute each vector's length
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)

    # Normalize the dot product so the result stays between about -1 and 1
    return dot / (norm1 * norm2)

sim_ab = cosine_sim(emb_a, emb_b)
sim_ac = cosine_sim(emb_a, emb_c)

print(f"SSO question vs SSO help:  {sim_ab:.4f}")
print(f"SSO question vs pricing:   {sim_ac:.4f}")

SSO question vs SSO help:  0.2792
SSO question vs pricing:   0.1307


The two SSO sentences have a higher similarity score. The pricing sentence has a much lower score compared to the SSO sentence. This is exactly what we want. When someone asks about SSO, the search will return SSO documents, not pricing documents.


<Note type="important">

The quality of your embeddings controls the quality of your search. If the embedding model does not understand your domain terms (like "SAML" or "IdP"), it will match questions to the wrong documents. For specialized domains, test different embedding models on your actual data.

Here the open-source `all-MiniLM-L6-v2` model does a good job, but in some cases, you might need a more powerful model or a domain-specific one. n8n's flexibility allows you to choose the best embedding provider for your use case.

</Note>

## Step 4: vector database (storing and searching embeddings)

As our documents grow, we will have more and more chunks and embeddings. With 10 documents, we have 50 chunks. With 1,000 documents, we have 5,000 chunks. With 100,000 documents, we have 500,000 chunks. Searching through all of them every time would be too slow. A **vector database** stores embeddings and finds the most similar ones quickly. Without it, you would compare every query against every chunk one by one. That works for 50 chunks. It does not work for 500,000 chunks.

Vector databases use special indexing algorithms to find the closest matches in milliseconds, even with millions of vectors.

<Note type="info">

One common algorithm is **[HNSW](https://arxiv.org/abs/1603.09320)** (Hierarchical Navigable Small World). It organizes vectors into a graph structure so the database can find similar ones without checking every single entry. You do not need to understand how HNSW works internally. Just know that it is what makes vector search fast.

</Note>

We use **ChromaDB** here because it is simple and runs locally. In production, you might use **Qdrant** (fast, written in Rust, great filtering), **Pinecone** (cloud-managed), or **Supabase** (PostgreSQL with pgvector). n8n has built-in nodes for Qdrant, Pinecone, Supabase, and in-memory vector stores.

Let us store our CloudSync documents in ChromaDB and search them.

In [37]:
# Create a ChromaDB collection (like a table in a regular database)
client = chromadb.Client()

# Clean up if re-running
for col in client.list_collections():
    if col.name == "cloudsync":
        client.delete_collection("cloudsync")

collection = client.create_collection(
    name="cloudsync",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created")

ChromaDB collection created


`chromadb.Client()` creates a connection to a local, in-memory ChromaDB instance. This means the data lives only in RAM and disappears when the notebook stops. In production, you would use a persistent ChromaDB server or a cloud-hosted vector database.

The cleanup loop checks if a collection named `"cloudsync"` already exists. If it does, it deletes it. This prevents errors when you re-run the cell.

`client.create_collection(name="cloudsync", metadata={"hnsw:space": "cosine"})` creates a new collection. The `metadata` argument tells ChromaDB to use cosine similarity (the same metric we coded manually earlier) when comparing vectors.

In [38]:
# Store each document as a single chunk (documents are short enough)
# In a real system, you would chunk first, then store each chunk

ids = []
texts = []
metadatas = []
embeddings = []

for i, doc in enumerate(documents):
    doc_id = f"doc_{i}"
    ids.append(doc_id)
    texts.append(doc["content"])
    
    meta = {
        "title": doc["title"],
        "source": doc["source"]
    }
    metadatas.append(meta)
    
    emb = model.encode(doc["content"])
    embeddings.append(emb.tolist())

collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings
)

print(f"Stored {collection.count()} documents in ChromaDB")

Stored 5 documents in ChromaDB


Each entry in ChromaDB has four parts: 
- an ID, 
- the original text, 
- metadata (title and source), and 
- the embedding vector. 

Now we can search. To search, we convert the query into an embedding and ask ChromaDB for the most similar entries. The results are ranked by similarity score. The top results are the most relevant chunks for that question.

In [39]:
# Search: ask a question and find the most relevant documents

question = "How much does CloudSync cost?"
question_embedding = model.encode(question)
question_embedding_list = question_embedding.tolist()

results = collection.query(
    query_embeddings=[question_embedding_list],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

print(f"Question: '{question}'")
print()

result_docs = results["documents"][0]
result_metas = results["metadatas"][0]
result_dists = results["distances"][0]

for i in range(len(result_docs)):
    # Similarity is 1 - distance because distance is how far apart the vectors are, and similarity is how close they are.
    similarity = 1 - result_dists[i]
    title = result_metas[i]["title"]
    print(f"Result {i + 1} (similarity: {similarity:.4f})")
    print(f"  Title: {title}")
    print(f"  Text: {result_docs[i][:100]}...")
    print()

Question: 'How much does CloudSync cost?'

Result 1 (similarity: 0.4687)
  Title: CloudSync API basics
  Text: All API requests need an API key from Settings > API Keys. The base URL is https://api.cloudsync.io/...

Result 2 (similarity: 0.3753)
  Title: Pricing and plans
  Text: Starter plan: $9/user/month with 50GB storage. Business plan: $29/user/month with 500GB storage, API...



Look at the results carefully. The question was "How much does CloudSync cost?" The best answer is clearly in the Pricing and plans document. But that document came back as Result 2, not Result 1. The top result is CloudSync API basics, which talks about rate limits, not prices.
Why did the embedding model get the ranking wrong? Because the API basics document contains the words "Business plans" and "Starter plans" in the context of rate limits. It also contains "CloudSync" in the URL. The embedding model sees word overlap between "How much does CloudSync cost?" and those terms. So it scores the API document higher than the pricing document.
The pricing document does appear in the results. It is just not ranked first.
This is a common problem in RAG systems. Embedding models match on meaning and on surface patterns like shared words. The top result is not always the best result. If you only retrieve 1 document, you might miss the correct answer entirely.

> Never trust the top-1 result blindly. Always retrieve more chunks than you think you need. The correct answer might be at position 2 or 3.

The fix is simple. Retrieve more documents. If you retrieve 3 or 4 instead of 2, the correct document will almost always appear somewhere in the results. The LLM is good at reading all the retrieved chunks and finding the right information, even if the ranking is imperfect.
Let us test this. We ask the same question but retrieve 4 results instead of 2.

In [40]:
# Retrieve more documents to handle imperfect ranking
# Same question, but n_results=4 instead of 2

question = "How much does CloudSync cost?"
question_embedding = model.encode(question)
question_embedding_list = question_embedding.tolist()

results_broad = collection.query(
    query_embeddings=[question_embedding_list],
    n_results=4,
    include=["documents", "metadatas", "distances"]
)

broad_docs = results_broad["documents"][0]
broad_metas = results_broad["metadatas"][0]
broad_dists = results_broad["distances"][0]

print(f"Question: '{question}'")
print(f"Retrieved {len(broad_docs)} documents:\n")

for i in range(len(broad_docs)):
    similarity = 1 - broad_dists[i]
    title = broad_metas[i]["title"]
    print(f"  Result {i + 1} (similarity: {similarity:.4f}) — {title}")

Question: 'How much does CloudSync cost?'
Retrieved 4 documents:

  Result 1 (similarity: 0.4687) — CloudSync API basics
  Result 2 (similarity: 0.3753) — Pricing and plans
  Result 3 (similarity: 0.2381) — SSO setup with SAML
  Result 4 (similarity: 0.1293) — Escalation procedures


The pricing document is still in the results. It did not disappear. It just was not at position 1. By retrieving 4 documents instead of 2, you guarantee that the correct chunk appears in the context, even if the ranking is not perfect.
When the LLM receives all 4 chunks in the prompt, it reads through them and picks the relevant information. It does not care about the ranking order. It reads everything and answers from the best match.

<Note type="important">
In production, retrieve 3 to 5 chunks for most use cases. Retrieving too few risks missing the answer. Retrieving too many adds noise and increases cost (more tokens in the prompt). Start with 3 and adjust based on testing.
</Note>

This leads to an important design rule. **Retrieval does not need to be perfect.** It needs to be good enough. The vector search narrows 500,000 documents down to 4 candidates. The LLM reads those 4 candidates and finds the answer. The search handles volume. The LLM handles precision. They work as a team.

In [41]:
# Metadata filtering: restrict search to only help center articles
# This is how you prevent internal documents from leaking to customers

question_2 = "When should I escalate a support ticket?"
emb_2 = model.encode(question_2)

# WITHOUT filter (returns internal runbook)
results_all = collection.query(
    query_embeddings=[emb_2.tolist()],
    n_results=1,
    include=["metadatas"]
)

# WITH filter (only help_center docs)
results_filtered = collection.query(
    query_embeddings=[emb_2.tolist()],
    n_results=1,
    include=["metadatas"],
    where={"source": "help_center"}
)

top_all = results_all["metadatas"][0][0]
top_filtered = results_filtered["metadatas"][0][0]

print(f"Question: '{question_2}'")
print(f"Without filter -> {top_all['title']} (source: {top_all['source']})")
print(f"With filter    -> {top_filtered['title']} (source: {top_filtered['source']})")

Question: 'When should I escalate a support ticket?'
Without filter -> Escalation procedures (source: internal_runbook)
With filter    -> Pricing and plans (source: help_center)


Without the filter, the internal escalation runbook appears. With `source: help_center`, it is excluded. In n8n, you set these filters on the vector store retriever node. This is your **access control** layer.

<Note type="important">

Metadata filtering is a security requirement, not a nice-to-have. A public chatbot must never return internal documents. Always set source filters based on who is asking the question.

</Note>

## Step 5: generation (the LLM reads and answers)

The last step is **generation**. You take the retrieved documents, put them into a prompt together with the user's question, and send everything to the LLM. The model reads the context and writes an answer based on that context. 

CPF stands for **Context, Prompt, Format**. The retrieved documents are the **context**. The user's question is part of the **prompt**. You also add instructions to the prompt, like 

> "Answer the question based only on the following documents. If the answer is not in the documents, say you don't know." 

Finally, you specify the **format** of the answer, for example "Write a short answer in 2 sentences."

In [42]:
# Build a RAG prompt using the CPF (Context Prompt Format)

question = "What is the price of the Business plan?"

# Step A: Retrieve the top 2 documents from the vector database
question_embedding = model.encode(question)
question_embedding_list = question_embedding.tolist()

results = collection.query(
    query_embeddings=[question_embedding_list],
    n_results=2,
    include=["documents", "metadatas"]
)

# Step B: Format each retrieved document as a labeled context block
retrieved_docs = results["documents"][0]
retrieved_metas = results["metadatas"][0]

context_parts = []
for i in range(len(retrieved_docs)):
    title = retrieved_metas[i]["title"]
    entry = f"[Source: {title}]\n{retrieved_docs[i]}"
    context_parts.append(entry)

context = "\n\n---\n\n".join(context_parts)

# Step C: Assemble the prompt using the CPF structure
# CPF = instructions + context + question
system_instruction = (
    "You are a CloudSync support assistant.\n"
    "Answer ONLY from the context below. "
    "If the answer is not in the context, say \"I don't know.\"\n"
    "Keep your answer short and direct."
)

user_prompt = f"""CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

print("=== SYSTEM INSTRUCTION ===")
print(system_instruction)
print()
print("=== USER PROMPT (context + question) ===")
print(user_prompt)

=== SYSTEM INSTRUCTION ===
You are a CloudSync support assistant.
Answer ONLY from the context below. If the answer is not in the context, say "I don't know."
Keep your answer short and direct.

=== USER PROMPT (context + question) ===
CONTEXT:
[Source: Pricing and plans]
Starter plan: $9/user/month with 50GB storage. Business plan: $29/user/month with 500GB storage, API access, and SSO. Enterprise plan: custom pricing with unlimited storage. Annual billing is default. Monthly billing costs 20% more. Educational institutions get 40% off the Business plan.

---

[Source: CloudSync API basics]
All API requests need an API key from Settings > API Keys. The base URL is https://api.cloudsync.io/v2/. Rate limits are 1000 requests per minute for Business plans and 100 per minute for Starter plans. If you exceed the limit, you get HTTP 429.

QUESTION: What is the price of the Business plan?

ANSWER:


Here is what each part of the code does.

**Step A** converts the user's question into an embedding and searches ChromaDB for the 2 most similar documents. This is the retrieval step you learned in Step 4.

**Step B** formats each retrieved document into a labeled block. The `[Source: Pricing and plans]` label tells the model where the information comes from. The `---` separator between blocks makes it easy for the model to distinguish one source from another.

**Step C** assembles the final prompt using the CPF structure. The `system_instruction` tells the model its role and rules. The `user_prompt` contains the context and the question. Notice the structure: context first, then the question, then "ANSWER:" to signal the model should start responding.

The prompt is ready. Now we send it to the OpenAI API.

In [43]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Set your API key for hands-on sections

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("OpenAI client ready")

OpenAI client ready


In [44]:
# Send the CPF prompt to the OpenAI API
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_prompt}
    ],
    # Low temperature for more factual, less creative answers
    temperature=0.1
)

# Extract the answer from the response object
answer = response.choices[0].message.content

print(f"Question: {question}")
print(f"Answer:   {answer}")

Question: What is the price of the Business plan?
Answer:   The price of the Business plan is $29/user/month.


<Note type="important">

The `system` message and the `user` message serve different purposes. The system message defines **rules** that apply to every answer. The user message contains the **specific question and its context**. In n8n, the system message goes into the "System Message" field of the AI Agent node. The user message is built automatically from the retriever output and the user's input.

</Note>

### Putting it together: a complete RAG function

Let us wrap the entire pipeline (retrieve + format + generate) into one function. This makes it easy to test different questions.

In [ ]:
def ask_cloudsync(question, source_filter=None):
    """Complete RAG pipeline: retrieve context, build CPF prompt, generate answer."""

    # Step 1: Embed the question
    q_embedding = model.encode(question)
    q_embedding_list = q_embedding.tolist()

    # Step 2: Search the vector database
    search_params = {
        "query_embeddings": [q_embedding_list],
        "n_results": 2,
        "include": ["documents", "metadatas"]
    }

    # Add source filter if provided (e.g., only help_center docs)
    if source_filter is not None:
        search_params["where"] = {"source": source_filter}

    results = collection.query(**search_params)

    # Step 3: Format retrieved documents into context blocks
    docs = results["documents"][0]
    metas = results["metadatas"][0]

    context_parts = []
    for i in range(len(docs)):
        title = metas[i]["title"]
        entry = f"[Source: {title}]\n{docs[i]}"
        context_parts.append(entry)

    context = "\n\n---\n\n".join(context_parts)

    # Step 4: Build the CPF prompt
    system_msg = (
        "You are a CloudSync support assistant.\n"
        "Answer ONLY from the context below. "
        "If the answer is not in the context, say \"I don't know.\"\n"
        "Keep your answer short and direct."
    )

    user_msg = f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:"

    # Step 5: Send to the OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        # Low temperature for more factual, less creative answers
        temperature=0.1
    )

    answer = response.choices[0].message.content
    return answer

In [46]:
# Test the full RAG pipeline with different questions

test_questions = [
    "What is the price of the Business plan?",
    "How do I set up SSO with Okta?",
    "What happens if I exceed the API rate limit?",
    "What is the weather in Tokyo?"
]

for q in test_questions:
    answer = ask_cloudsync(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

Q: What is the price of the Business plan?
A: The price of the Business plan is $29/user/month.

Q: How do I set up SSO with Okta?
A: Create a SAML app in Okta and use https://auth.cloudsync.io/saml/callback as the ACS URL. Then, upload the IdP metadata XML in Settings > Security > SSO.

Q: What happens if I exceed the API rate limit?
A: You get HTTP 429.

Q: What is the weather in Tokyo?
A: I don't know.



The first three questions are about CloudSync. The model should answer each one using the retrieved documents. The pricing question should return "$29/user/month." The SSO question should mention SAML and Okta. The rate limit question should mention HTTP 429.

The last question ("What is the weather in Tokyo?") is a test. No CloudSync document talks about weather. If the CPF is working correctly, the model should respond with "I don't know" instead of making up an answer. This is the **grounding** behavior that makes RAG reliable. The model only answers from what it was given. It does not guess.

<Note type="tip">

Always test your RAG system with at least one question that is completely off-topic. If the model answers it confidently, your CPF instructions are too weak. Strengthen the system instruction or lower the temperature.

</Note>

### Testing with source filters

Remember the metadata filter from Step 4? You can use it inside the full pipeline. This is how you build a **public chatbot** that never leaks internal documents.

In [47]:
# Public chatbot: only uses help_center documents
public_answer = ask_cloudsync(
    "When should I escalate a support ticket?",
    source_filter="help_center"
)

print("Public chatbot (help_center only):")
print(public_answer)
print()

# Internal tool: uses all documents including runbooks
internal_answer = ask_cloudsync(
    "When should I escalate a support ticket?"
)

print("Internal tool (all documents):")
print(internal_answer)

Public chatbot (help_center only):
I don't know.

Internal tool (all documents):
You should escalate a support ticket to Tier 2 for data loss or API 500 errors, and to Tier 3 if more than 10 users are affected or if an Enterprise customer reports SLA downtime.


The public chatbot can only see help center articles. It does not have access to the internal escalation runbook. So it will either give a general answer from what it found, or say "I don't know."

The internal tool sees all documents, including the runbook. It can give a detailed answer about Tier 1, Tier 2, and Tier 3 escalation rules.

Same question, same pipeline, same CPF. The only difference is the source filter. This is a simple but powerful access control mechanism.

<Note type="important">

Metadata filtering is a security requirement, not a nice-to-have. A public chatbot must never return internal documents. Always set source filters based on who is asking the question.

</Note>

<Note type = "tip" title ="CPF variations">

The CPF you built above is the simplest version. In production, you might adjust the format depending on the use case. Here are three common variations.

**Strict CPF** 

The system instruction says: "If the context does not contain the answer, say 'I don't have enough information to answer this.' Do not use any knowledge from outside the context." This is for customer-facing chatbots where hallucination is unacceptable.

**Conversational CPF** 

The system instruction says: "Use the context to inform your answer, but you may also draw on your general knowledge if the context is insufficient. Always indicate which parts come from the context and which are general knowledge." This is for internal research tools where flexibility matters more than strict grounding.

**Citation CPF** 

The system instruction says: "After each claim, cite the source in square brackets, like [Source: Pricing and plans]. If a claim is not supported by any source, do not include it." This is for compliance and audit scenarios where every answer must be traceable.

The structure stays the same (instructions + context + question). Only the rules change.

In n8n, the CPF is split across two fields in the AI Agent node. The **system message** field holds the instructions. The **retriever** node provides the context automatically. The **user input** provides the question. n8n assembles the CPF for you.

</Note>

## Resources

- [n8n AI documentation](https://docs.n8n.io/advanced-ai/)
- [OpenAI API reference](https://platform.openai.com/docs/api-reference/chat)
- [OpenAI pricing](https://openai.com/api/pricing/) 
- [ChromaDB documentation](https://docs.trychroma.com)
- [Sentence Transformers](https://www.sbert.net)
- [Qdrant documentation](https://qdrant.tech/documentation)